# Official on Paper, Spoken on the Ground

Is the legally official language always the most widely spoken? This notebook
compares CLDR official-language status with per-country speaker-count rankings.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [1]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [2]:
db = low.LanguagesOfTheWorld()
print(db)

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)


## 2 · Official language fields

Each `Country` exposes three CLDR-derived lists:
- `official_languages` — nationally official
- `official_regional_languages` — regionally official
- `de_facto_official_languages` — widely used without formal status

In [3]:
ch = db.countries.get("CH")
print(f"{ch.label} ({ch.code})")
print(f"  Official:          {[l.label for l in ch.official_languages]}")
print(f"  Regional official: {[l.label for l in ch.official_regional_languages]}")
print(f"  De facto official: {[l.label for l in ch.de_facto_official_languages]}")

Switzerland (CH)
  Official:          ['German', 'French', 'Italian']
  Regional official: ['Romansh']
  De facto official: ['Swiss German']


## 3 · Top spoken language per country (CLDR source)

In [4]:
def top_spoken_language(country):
    cldr = [sc for sc in country.speaker_counts if sc.source == "cldr" and sc.speaker_count]
    if not cldr:
        return None, 0
    top = max(cldr, key=lambda sc: sc.speaker_count)
    return top.language, top.speaker_count

rows = []
for country in db.countries:
    top_lang, top_count = top_spoken_language(country)
    official = {l.part3 for l in country.official_languages}
    de_facto = {l.part3 for l in country.de_facto_official_languages}
    regional = {l.part3 for l in country.official_regional_languages}
    top_part3 = top_lang.part3 if top_lang else None
    rows.append({
        "country": country.label,
        "code": country.code,
        "continent": country.continent.label,
        "official_count": len(official),
        "regional_count": len(regional),
        "de_facto_count": len(de_facto),
        "top_spoken": top_lang.label if top_lang else None,
        "top_spoken_count": top_count,
        "top_is_official": top_part3 in official if top_part3 else None,
        "top_is_de_facto": top_part3 in de_facto if top_part3 else None,
        "official_labels": ", ".join(l.label for l in country.official_languages),
    })

df = pd.DataFrame(rows)
df.head(10)

,country,code,continent,official_count,regional_count,de_facto_count,top_spoken,top_spoken_count,top_is_official,top_is_de_facto,official_labels
0,Andorra,AD,Europe,1,0,0,Catalan,43538,True,False,Catalan
1,United Arab Emirates,AE,Asia,1,0,0,Arabic,7825116,True,False,Arabic
2,Afghanistan,AF,Asia,2,2,0,Persian,20060800,True,False,"Persian, Pushto"
3,Antigua and Barbuda,AG,Americas,1,0,0,English,88265,True,False,English
4,Anguilla,AI,Americas,1,0,0,English,18445,True,False,English
5,Albania,AL,Europe,1,0,0,Albanian,3107100,True,False,Albanian
6,Armenia,AM,Asia,1,0,0,Armenian,2947002,True,False,Armenian
7,Angola,AO,Africa,1,0,0,Portuguese,24925407,True,False,Portuguese
8,Argentina,AR,Americas,1,0,0,Spanish,46994400,True,False,Spanish
9,American Samoa,AS,Oceania,1,0,1,Samoan,43456,True,False,Samoan


## 4 · Bar chart — official language count per country

In [5]:
multi = df[df["official_count"] > 0].sort_values("official_count", ascending=False).head(20)

fig = px.bar(
    multi,
    x="official_count",
    y="country",
    orientation="h",
    color="continent",
    text="official_count",
    labels={"official_count": "Official languages", "country": ""},
    title="Countries with the Most Official Languages",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=550)
fig.update_traces(textposition="outside")
fig.show()

## 5 · Mismatch countries

Countries where the most-spoken language (CLDR) is not nationally official.

In [6]:
mismatch = df[(df["top_is_official"] == False) & df["top_spoken"].notna()].sort_values("top_spoken_count", ascending=False)
print(f"Countries where top-spoken language is not official: {len(mismatch)}")
mismatch[["country", "top_spoken", "top_spoken_count", "official_labels"]].head(15)

Countries where top-spoken language is not official: 38


,country,top_spoken,top_spoken_count,official_labels
230,United States of America,English,328284480,
155,Mexico,Spanish,108514200,
67,Ethiopia,English,50976500,Amharic
60,Algeria,Algerian Arabic,39028675,"Arabic, French"
135,Morocco,Moroccan Arabic,32527212,"Arabic, Central Atlas Tamazight"
194,Sudan,Sudanese Arabic,30785053,"Arabic, English"
11,Australia,English,25697856,
210,Syrian Arab Republic,Levantine Arabic,20285590,Arabic
80,Ghana,Akan,13489749,English
203,Senegal,Wolof,13193250,French


## 6 · Case studies

In [7]:
for code in ("CH", "IN", "ZA", "US"):
    c = db.countries.get(code)
    top_lang, top_count = top_spoken_language(c)
    print(f"\n{c.label}")
    print(f"  Official: {[l.label for l in c.official_languages]}")
    print(f"  Top spoken (CLDR): {top_lang.label if top_lang else '—'} ({top_count:,})")
    cldr_sorted = sorted(
        [sc for sc in c.speaker_counts if sc.source == "cldr"],
        key=lambda sc: sc.speaker_count,
        reverse=True,
    )[:5]
    for sc in cldr_sorted:
        tag = ""
        if sc.language in c.official_languages:
            tag = " [official]"
        elif sc.language in c.de_facto_official_languages:
            tag = " [de facto]"
        print(f"    {sc.language.label}: {sc.speaker_count:,}{tag}")


Switzerland
  Official: ['German', 'French', 'Italian']
  Top spoken (CLDR): German (6,734,033)
    German: 6,734,033 [official]
    Swiss German: 5,847,976 [de facto]
    English: 3,987,256
    French: 3,455,622 [official]
    Italian: 1,329,085 [official]

India
  Official: ['English', 'Hindi']
  Top spoken (CLDR): Hindi (577,743,300)
    Hindi: 577,743,300 [official]
    English: 267,734,700 [official]
    Bengali: 114,139,530
    Telugu: 101,457,360
    Marathi: 98,639,100

South Africa
  Official: ['English']
  Top spoken (CLDR): English (18,737,206)
    English: 18,737,206 [official]
    Zulu: 14,506,224
    Xhosa: 10,879,668
    Afrikaans: 7,857,538
    Pedi: 5,681,604

United States of America
  Official: []
  Top spoken (CLDR): English (328,284,480)
    English: 328,284,480 [de facto]
    Spanish: 32,828,448
    Chinese: 2,359,544
    French: 1,914,992
    German: 1,607,226


## 7 · Summary

In [8]:
has_official = (df["official_count"] > 0).sum()
aligned = (df["top_is_official"] == True).sum()
print(f"Countries with ≥1 official language: {has_official}")
print(f"Countries where top-spoken is official: {aligned}")
print(f"Countries with de facto official languages: {(df['de_facto_count'] > 0).sum()}")

Countries with ≥1 official language: 236
Countries where top-spoken is official: 209
Countries with de facto official languages: 20
